# Generating the ESM2 embeddings

Was originally named "embedding_sequence.ipynb"

# Dependencias

In [1]:
import torch
import warnings
import sqlite3
import os
import yaml

from Bio import SeqIO, BiopythonWarning
from Bio.PDB import PDBList, PDBParser
from Bio.PDB.PDBExceptions import PDBConstructionWarning
from transformers import AutoTokenizer, AutoModel, EsmTokenizer, EsmModel

warnings.filterwarnings('ignore', category=PDBConstructionWarning)

c:\Users\BISITE\Desktop\GAN-epitope-prediction\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Config

In [14]:
# Load configuration
# use config data to load model and tokenizer
with open("config/config.yaml", 'r') as f:
    config = yaml.safe_load(f)

tokenizer = AutoTokenizer.from_pretrained(config['model_embedding'])
model = AutoModel.from_pretrained(config['model_embedding'])

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
# Check for GPU availability and print details
if torch.cuda.is_available():
    print("✅ GPU detectada:")
    print(f"Nombre: {torch.cuda.get_device_name(0)}")
    print(f"Memoria total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print(" No se detectó GPU, usando CPU.")

# Test device assignment
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

✅ GPU detectada:
Nombre: NVIDIA RTX A4000
Memoria total: 17.17 GB
Using device: cuda


# Functions

Tenemos una función principal: 
- **process_database** : Recibe una ruta de la base de datos y extrae el identificador y la secuencia:
  1) Extrae el embedding de la secuencia (*extract_residue_embeddings*)
  2) Guarda este embedding junto a la secuencia en la ruta propuesta (data/embeddings) (*save_embedding_with_residues*)

Y dos funciones complementarias:
- **extract_residue_embeddings**
- **save_embeddings_with_residues**
### English
We have one main function:
- **process_database**: Receives a database path and extracts identifiers and sequences:
  1) Extracts sequence embeddings (*extract_residue_embeddings*)
  2) Saves these embeddings along with the sequence to the specified path (data/embeddings) (*save_embedding_with_residues*)

And two supporting functions:
- **extract_residue_embeddings**: Generates per-residue embeddings using the ESM-2 model
- **save_embedding_with_residues**: Stores embeddings and residue sequences in .pt files

In [16]:
def save_embedding_with_residues(output_dir, protein_id, embeddings, residues):
    """
    Saves embeddings and residue sequence to a file.
    
    Args:
        output_dir (str): Directory where files will be saved.
        protein_id (str): Protein identifier (PDB or UniProt ID).
        embeddings (torch.Tensor): Tensor with embeddings, shape [num_residues, embedding_dim].
        residues (str): Amino acid sequence (residues).
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    filepath = os.path.join(output_dir, f"{protein_id}_embedding.pt")
    
    # Save as dictionary containing sequence and embeddings
    torch.save({
        "sequence": list(residues),  # Convert string to list of amino acids
        "embeddings": embeddings      # Tensor of shape [seq_length, embedding_dim]
    }, filepath)
    print(f"Saved embeddings to {filepath}")


def extract_residue_embeddings(sequence):
    """
    Extracts per-residue embeddings using the HuggingFace ESM-2 model.
    
    Args:
        sequence (str): Amino acid sequence as string (without spaces).
    
    Returns:
        torch.Tensor: Per-residue embeddings, shape [num_residues, embedding_dim]
    """
    # Tokenize sequence and add special tokens (BOS/EOS)
    inputs = tokenizer(sequence, return_tensors="pt", add_special_tokens=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Move inputs and model to GPU if available
    inputs = {k: v.to(device) for k, v in inputs.items()}
    model.to(device)
    
    # Generate embeddings without computing gradients (inference mode)
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Extract last hidden state and remove batch dimension
    embeddings = outputs.last_hidden_state.squeeze(0)
    
    # Remove special tokens (BOS and EOS) by slicing [1:-1]
    # Move back to CPU for saving
    residue_embeddings = embeddings[1:-1, :].cpu()
    
    # Validate that embedding length matches sequence length
    if residue_embeddings.shape[0] != len(sequence):
        print(f"FATAL ERROR: Slicing resulted in {residue_embeddings.shape[0]} embeddings for a {len(sequence)} residue sequence.")
        raise RuntimeError("Embedding slicing failed. Check model setup.")
    
    return residue_embeddings


def filter_id(identifier):
    """
    Extracts clean protein ID from full URL or reference.
    
    Converts: http://www.ncbi.nlm.nih.gov/protein/P08176.2 → P08176.2
    
    Args:
        identifier (str): Full protein reference (may include URL).
    
    Returns:
        str: Clean protein ID.
    """
    if '/' in identifier:
        return identifier.split('/')[-1]
    return identifier


def process_database(dbpath, table, output_dir="data/embeddings"):
    """
    Processes protein sequences from a SQLite table with columns 'Protein' and 'sequences'.
    
    Pipeline:
        1. Connects to SQLite database
        2. Queries protein names (Protein) and sequences
        3. Generates ESM-2 embeddings for each sequence
        4. Saves embeddings with sequences to .pt files
    
    Args:
        dbpath (str): Path to SQLite database file.
        table (str): Name of table containing 'Protein' and 'sequences' columns.
        output_dir (str): Directory where embedding files will be saved.
    
    Note:
        Assumes database has 'Protein' and 'sequences' columns.
    """
    # Connect to database and retrieve protein names + sequences
    conn = sqlite3.connect(dbpath)
    query = 'SELECT Protein, sequences FROM {}'.format(table)
    rows = conn.execute(query).fetchall()
    
    # Process each protein sequence
    for idx, (protein_id, sequence) in enumerate(rows):
        try:
            # Generate embeddings (convert to uppercase for consistency)
            embeddings = extract_residue_embeddings(sequence.upper())
            
            # Clean protein ID (remove URL prefix if present)
            protein_id = filter_id(protein_id)
            
            # Log progress
            print(f"Processed {idx+1}/{len(rows)}: {protein_id},\nsequence: {sequence},\nsequence length: {len(sequence)}, embeddings shape: {embeddings.shape}")
            
            # Save embeddings and sequence to file
            save_embedding_with_residues(output_dir, protein_id, embeddings, sequence)
            
        except Exception as e:
            print(f"Error processing {protein_id}: {e}")

In [17]:
db_path = "data/db_epitopes.db"
table = "Proteins"

conn = sqlite3.connect(db_path)
query = 'SELECT Protein, sequences FROM {}'.format(table)
rows = conn.execute(query).fetchall()
for idx, (protein_id, sequence) in enumerate(rows):
    print(f"{idx+1}/{len(rows)}: {protein_id}, sequence length: {len(sequence)}")    

1/1178: P08176.2, sequence length: 320
2/1178: AAO20853.1, sequence length: 341
3/1178: P26012.1, sequence length: 769
4/1178: Q07820.3, sequence length: 350
5/1178: AAA74751.1, sequence length: 582
6/1178: ABL74395.1, sequence length: 540
7/1178: P10515.3, sequence length: 647
8/1178: AFJ80778.1, sequence length: 193
9/1178: P0DTC2.1, sequence length: 1273
10/1178: Q14973.1, sequence length: 349
11/1178: P63096.2, sequence length: 354
12/1178: P06156.1, sequence length: 553
13/1178: AGL76709.1, sequence length: 501
14/1178: P03101.3, sequence length: 505
15/1178: XP_001348015.1, sequence length: 622
16/1178: AAN32775.1, sequence length: 495
17/1178: J7RUA5.1, sequence length: 1053
18/1178: AAB34852.1, sequence length: 417
19/1178: XP_001349859.1, sequence length: 1210
20/1178: ACP41953.1, sequence length: 566
21/1178: P08669.1, sequence length: 491
22/1178: P21980.2, sequence length: 687
23/1178: Q9NZQ7.1, sequence length: 290
24/1178: P16586.1, sequence length: 855
25/1178: P01024.2,

## Execute Main

In [18]:
db_path = "data/db_epitopes.db"
table = "Proteins"

process_database(db_path, table)

Processed 1/1178: P08176.2,
sequence: mkivlaiasllalsavyarpssiktfeeykkafnksyatfedeeaarknflesvkyvqsnggainhlsdlsldefknrflmsaeafehlktqfdlnaetnacsingnapaeidlrqmrtvtpirmqggcgscwafsgvaatesaylayrnqsldlaeqelvdcasqhgchgdtiprgieyiqhngvvqesyyryvareqscrrpnaqrfgisnycqiyppnvnkirealaqthsaiaviigikdldafrhydgrtiiqrdngyqpnyhavnivgysnaqgvdywivrnswdtnwgdngygyfaanidlmmieeypyvvil,
sequence length: 320, embeddings shape: torch.Size([320, 1280])
Saved embeddings to data/embeddings\P08176.2_embedding.pt
Processed 2/1178: AAO20853.1,
sequence: gshsmryfftsvsrpgrgeprfiavgyvddtqfvrfdsdaasqrmeprapwieqegpeywdgetrkvkahsqthrvdlgtlrgyynqseagshtvqrmygcdvgsdwrflrgyhqyaydgkdyialkedlrswtaadmaaqttkhkweaahvaeqlraylegtcvewlrrylengketlqrtdapkthmthhavsdheatlrcwalsfypaeitltwqrdgedqtqdtelvetrpagdgtfqkwaavvvpsgqeqrytchvqheglpkpltlrwepssqptipivgiiaglvlfgavitgavvaavmwrrkssdrkggsysqaassdsaqgsdvsltackv,
sequence length: 341, embeddings shape: torch.Size([341, 1280])
Saved embeddings to data/embeddings\AAO20853.1_embedding.pt
Processed 3

In [19]:
def load_embeddings_dict_from_files(folder_path):
    """
    Loads all .pt embedding files from a folder and generates a dictionary
    mapping each protein to a residue-to-embedding dictionary.

    Args:
        folder_path (str): Folder where .pt embedding files are stored.

    Returns:
        dict: Nested dictionary structure:
            {
                'protein_id1': {'A': tensor_embedding, 'K': tensor_embedding, ...},
                'protein_id2': {'M': tensor_embedding, 'V': tensor_embedding, ...},
                ...
            }
            where each protein maps to a dict of {residue: embedding_tensor}
    """
    embedding_dict = {}
    
    # Get all .pt files in the folder
    files = [f for f in os.listdir(folder_path) if f.endswith(".pt")]

    for filename in files:
        # Load embedding file
        filepath = os.path.join(folder_path, filename)
        data = torch.load(filepath)
        
        # Extract sequence (list of amino acids) and embeddings (tensor)
        sequence = data["sequence"]      # List: ['M', 'K', 'V', 'L', ...]
        embeddings = data["embeddings"]  # Tensor: shape [seq_length, embedding_dim]

        # Create residue -> embedding mapping for this protein
        # Maps each amino acid to its corresponding embedding vector
        residue_to_emb = {residue: embeddings[i] for i, residue in enumerate(sequence)}

        # Extract protein ID from filename (remove .pt extension)
        protein_id = os.path.splitext(filename)[0]
        
        # Store in main dictionary
        embedding_dict[protein_id] = residue_to_emb

    return embedding_dict

## Test

In [20]:
# Example usage: Load embeddings preserving sequence order
folder = "data/embeddings"
embedding_dict = {}

# Get all .pt embedding files from the folder
files = [f for f in os.listdir(folder) if f.endswith(".pt")]

for filename in files[0:1]:
    # Load each embedding file
    filename = "1A2Y_C_embedding.pt"
    filepath = f"data/embeddings/{filename}"
    data = torch.load(filepath)
    
    # Extract sequence as list (preserves order) and embeddings tensor
    sequence = data["sequence"]      # List to preserve residue order
    embeddings = data["embeddings"]  # Tensor: [seq_length, embedding_dim]

    # Extract protein ID from filename (remove .pt extension)
    protein_id = os.path.splitext(filename)[0]
    
    # Store with structure that maintains order and actual sequence length
    # This format is better for sequential processing (GNNs, transformers, etc.)
    embedding_dict[protein_id] = {
        "residues": sequence,       # Ordered list of amino acids
        "embeddings": embeddings    # Aligned embedding tensor
    }

# Summary statistics
print(f"Number of proteins loaded: {len(embedding_dict)}")

# Access first protein as example
first_protein = next(iter(embedding_dict))
residues = embedding_dict[first_protein]["residues"]
embeddings = embedding_dict[first_protein]["embeddings"]

# Display information about first protein
print(f"First protein: {first_protein}")
print(f"Number of residues: {len(residues)}")
print(f"Embeddings shape: {embeddings.shape}")  # Should be [len(residues), embedding_dim]

Number of proteins loaded: 1
First protein: 1A2Y_C_embedding
Number of residues: 129
Embeddings shape: torch.Size([129, 1280])


C:\Users\BISITE\AppData\Local\Temp\ipykernel_7220\4046737994.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(filepath)


In [21]:
# Print each residue with its corresponding embedding shape
for i, residue in zip(residues, embeddings):
    print(f"Residuo: {i}, Embedding shape: {residue}")

Residuo: k, Embedding shape: tensor([-0.0259, -0.1346, -0.1572,  ...,  0.1396,  0.1508, -0.1979])
Residuo: v, Embedding shape: tensor([ 0.1171,  0.0234,  0.1526,  ..., -0.1459,  0.3636,  0.2705])
Residuo: f, Embedding shape: tensor([-0.1586, -0.1348, -0.0305,  ...,  0.1402, -0.1960,  0.1212])
Residuo: g, Embedding shape: tensor([ 0.1092, -0.2197, -0.1414,  ..., -0.0398, -0.0894, -0.2016])
Residuo: r, Embedding shape: tensor([ 0.1971, -0.1372,  0.1194,  ..., -0.1222,  0.0019, -0.0196])
Residuo: c, Embedding shape: tensor([-0.2751,  0.0897, -0.0363,  ..., -0.3134, -0.3363,  0.0290])
Residuo: e, Embedding shape: tensor([ 0.3603, -0.2020, -0.0980,  ..., -0.2442,  0.0154, -0.0933])
Residuo: l, Embedding shape: tensor([ 0.2828, -0.2250,  0.2560,  ..., -0.2150,  0.1613, -0.2319])
Residuo: a, Embedding shape: tensor([-0.1190, -0.0976, -0.2441,  ..., -0.1129,  0.0040,  0.0610])
Residuo: a, Embedding shape: tensor([ 0.1168,  0.1495, -0.1416,  ..., -0.0477, -0.0211, -0.1507])
Residuo: a, Embeddin